# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I'm re-ranking using my logistic regression model's **honest** (held-out, client-grouped-split) test-set probabilities.

**Important scope note:** this queue only covers the ~30% of eligible pages that landed in the honest test split (`GroupShuffleSplit`, `test_size=0.3`, `random_state=42`) — not the full dataset the Week 4 baseline ranked. Scoring the training rows would defeat the point of holding them out.

I kept the Week 4 reason codes to serve as extra information for a human reviewer, but they are **not** used as model inputs or used to train the model itself — the ranking itself comes only from `model_probability`. The reason codes just help to explain, in human readable terms, *why* a page looks risky.

Reason codes (a page can carry more than one):
- `high_visibility_at_risk`: at least 1,000 prior impressions and a weak prior position.
- `weak_position_signal`: prior average position is worse than 10.
- `low_prior_engagement`: prior engagement rate is below 30% when sessions are available.
- `low_click_through_rate`: prior clicks are low relative to prior impressions.
- `limited_prior_visibility`: fewer than 1,000 prior impressions, but still at least 100 and eligible.

This is a decision-support ranking, not a causal claim about what will happen to any given page.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load the same February-March dataset used in Weeks 4-6
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()

# Same feature prep as Week 5 / Week 6
X = dataframe[['prior_impressions', 'prior_clicks', 'prior_avg_position',
               'prior_sessions', 'prior_engagement_rate']].copy()
X['prior_avg_position'] = X['prior_avg_position'].fillna(999)
X['prior_sessions'] = X['prior_sessions'].fillna(0)
X['prior_engagement_rate'] = X['prior_engagement_rate'].fillna(0)
X['prior_ctr'] = (dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)).fillna(0)

model_features = ['prior_impressions', 'prior_clicks', 'prior_ctr',
                   'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
X = X[model_features]
y = dataframe['future_decline_label'].values
groups = dataframe['client_hash_id']

# Same honest, client-grouped split as Week 6 (random_state=42, test_size=0.3)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
X_test_scaled = scaler.transform(X.iloc[test_idx])

model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
model.fit(X_train_scaled, y[train_idx])

# Scored only the test set (honest, unseen data) for ranking and reason code generation
test_dataframe = dataframe.iloc[test_idx].copy()
test_dataframe['model_probability'] = model.predict_proba(X_test_scaled)[:, 1]

# Reason codes
test_dataframe['prior_ctr'] = X.iloc[test_idx]['prior_ctr']
test_dataframe['high_visibility_at_risk'] = (
    (test_dataframe['prior_impressions'] >= 1000) & (test_dataframe['prior_avg_position'] > 10)
).astype(int)
test_dataframe['weak_position_signal'] = (
    test_dataframe['prior_avg_position'] > 10
).fillna(False).astype(int)
test_dataframe['low_prior_engagement'] = (
    (test_dataframe['prior_sessions'] > 0) & (test_dataframe['prior_engagement_rate'] < 0.30)
).fillna(False).astype(int)
test_dataframe['low_click_through_rate'] = (
    test_dataframe['prior_ctr'] < 0.01
).fillna(False).astype(int)
test_dataframe['limited_prior_visibility'] = (
    test_dataframe['prior_impressions'] < 1000
).astype(int)

def make_reason_code(row):
    reasons = []
    if row['high_visibility_at_risk']:
        reasons.append('high_visibility_at_risk')
    if row['weak_position_signal']:
        reasons.append('weak_position_signal')
    if row['low_prior_engagement']:
        reasons.append('low_prior_engagement')
    if row['low_click_through_rate']:
        reasons.append('low_click_through_rate')
    if row['limited_prior_visibility']:
        reasons.append('limited_prior_visibility')
    return '; '.join(reasons) if reasons else 'no_flag_triggered'

test_dataframe['reason_code'] = test_dataframe.apply(make_reason_code, axis=1)

# Rank by the model's predicted probability (honest, unseen-data score)
ranked = test_dataframe.sort_values(
    ['model_probability', 'prior_impressions'],
    ascending=[False, True],
).reset_index(drop=True)
ranked['rank'] = np.arange(1, len(ranked) + 1)

output_path = repo_root / 'work' / 'outputs' / 'march_model_ranked_queue.csv'
ranked_columns = [
    'rank',
    'client_hash_id',
    'content_hash_id',
    'model_probability',
    'reason_code',
    'prior_impressions',
    'prior_clicks',
    'prior_ctr',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
    'future_impressions',
    'future_decline_label',
]
ranked[ranked_columns].to_csv(output_path, index=False)

top_20 = ranked.head(20)
precision_at_20 = top_20['future_decline_label'].mean()
base_rate = ranked['future_decline_label'].mean()

metrics = pd.DataFrame({
    'metric': ['test_set_rows', 'future_decline_base_rate', 'precision_at_20'],
    'value': [len(ranked), base_rate, precision_at_20],
})

print(f"Saved model-ranked queue to: {output_path}")
print(metrics)
ranked[ranked_columns].head(10)

Saved model-ranked queue to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\march_model_ranked_queue.csv
                     metric        value
0             test_set_rows  28904.00000
1  future_decline_base_rate      0.18769
2           precision_at_20      0.35000


,rank,client_hash_id,content_hash_id,model_probability,reason_code,prior_impressions,prior_clicks,prior_ctr,prior_avg_position,prior_sessions,prior_engagement_rate,future_impressions,future_decline_label
0,1,client_62f4a7e64f5e0096,content_90abf1c28b62a2e4,0.429304,weak_position_signal; low_click_through_rate; ...,113.0,0.0,0.0,104.716814,0.0,NaN,53.0,1
1,2,client_62f4a7e64f5e0096,content_1e9fa51c07fee766,0.405470,weak_position_signal; low_click_through_rate; ...,103.0,0.0,0.0,92.106796,0.0,NaN,139.0,0
2,3,client_62f4a7e64f5e0096,content_7e4133c46bb7c589,0.399165,weak_position_signal; low_click_through_rate; ...,124.0,0.0,0.0,88.798387,0.0,NaN,49.0,1
3,4,client_62f4a7e64f5e0096,content_3a398db04bf7c610,0.391199,weak_position_signal; low_click_through_rate; ...,128.0,0.0,0.0,84.531250,0.0,NaN,13.0,1
4,5,client_62f4a7e64f5e0096,content_7ab0876ca393025c,0.379690,weak_position_signal; low_click_through_rate; ...,133.0,0.0,0.0,78.308271,0.0,NaN,1088.0,0
5,6,client_62f4a7e64f5e0096,content_edb3d41e07485a15,0.378435,weak_position_signal; low_click_through_rate; ...,132.0,0.0,0.0,77.621212,0.0,NaN,201.0,0
6,7,client_62f4a7e64f5e0096,content_c2870dfa1296ea52,0.375297,weak_position_signal; low_click_through_rate; ...,335.0,0.0,0.0,76.456716,0.0,NaN,1644.0,0
7,8,client_62f4a7e64f5e0096,content_3b88098d80da16b5,0.374190,weak_position_signal; low_click_through_rate; ...,317.0,0.0,0.0,75.801262,0.0,NaN,822.0,0
8,9,client_62f4a7e64f5e0096,content_af174291be2b1a26,0.373435,weak_position_signal; low_click_through_rate; ...,229.0,0.0,0.0,75.148472,0.0,NaN,262.0,0
9,10,client_62f4a7e64f5e0096,content_1424a94ca74e2388,0.372419,weak_position_signal; low_click_through_rate; ...,148.0,0.0,0.0,74.371622,0.0,NaN,288.0,0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.